# 1. Preparation

In [1]:
import sys
sys.path.append('../../')
from model_001 import LucaQuadruple_final_dropout, fluProfiler_Config
from tqdm import tqdm
import os
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import load_embedding
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import pickle
from utilities import print_exams

In [2]:
class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def list2df(mylist, period):
    merged_rows = []
    for i in range(0, len(mylist), period):
        merged_row = []
        for j in range(period):
            merged_row += mylist[i + j]
        merged_rows.append(merged_row)

    return pd.DataFrame(merged_rows)

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask


In [29]:
device = torch.device('cuda:2')
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
new_columns = ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d','seq_a', 'seq_b', 'seq_c', 'seq_d', 
               'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'label']
data_path = '/data/chenyihao/dataset'

Crick_all = pd.read_csv(data_path + '/all.csv')
dataframe = Crick_all.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'seq_id_c': 'first', 'seq_id_d': 'first',
                                                  'seq_type_a': 'first', 'seq_type_b': 'first', 'seq_type_c': 'first', 'seq_type_d': 'first',
                                                  'serumName': 'first', 'virusName': 'first', 'label': 'mean'}).reset_index()
Crick_all_final = dataframe[new_columns]

Artificial_all = pd.read_csv(data_path + '/Artificial_data.csv')
dataframe = Artificial_all.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'seq_id_c': 'first', 'seq_id_d': 'first',
                                                       'label': 'mean'}).reset_index()

Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
Crick_serumType = pd.concat([Crick_H1N1,Crick_H3N2])[['virusName', 'serumType']].drop_duplicates(subset=['virusName']).reset_index(drop=True)
Crick_all_final = Crick_all_final.merge(right=Crick_serumType,how='left', left_on='virusName', right_on='virusName')

train_data, test_data = train_test_split(Crick_all_final, test_size=0.1, random_state=42)
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)
train_data = pd.concat([train_data, Artificial_all], axis=0)

train_dataset = fluProfiler_Dataset(train_data)
valid_dataset = fluProfiler_Dataset(valid_data)
test_dataset = fluProfiler_Dataset(test_data)

batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [30]:
H1N1_test_data = test_data[test_data['serumType'] == 'H1N1']
H3N2_test_data = test_data[test_data['serumType'] == 'H3N2']

H1N1_test_dataloader = DataLoader(fluProfiler_Dataset(H1N1_test_data), batch_size=batch_size, shuffle=False)
H3N2_test_dataloader = DataLoader(fluProfiler_Dataset(H3N2_test_data), batch_size=batch_size, shuffle=False)

In [32]:
H3N2_test_data['label'].value_counts()

label
 2.000000    646
 3.000000    492
 4.000000    450
 1.000000    390
 0.000000    309
            ... 
 1.857143      1
-0.066667      1
 0.888889      1
 4.800000      1
 2.514286      1
Name: count, Length: 184, dtype: int64

In [5]:
import os
from tqdm import tqdm

embedding_df = test_data
# load embedding
sequence_names = pd.concat([embedding_df['seq_id_a'],embedding_df['seq_id_b'],
                            embedding_df['seq_id_c'],embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding", files=sequence_names)
embeddings = [emb.to(device) for emb in embeddings]
emb_dict = dict(zip(IDs, embeddings))

Loading tensor: 100%|██████████| 6419/6419 [15:56<00:00,  6.71file/s]


In [12]:
model = torch.load(f='../../../model/model_001.pth', weights_only=False, map_location=device)

# 2.

In [3]:
CDC_all = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/dataset/CDC_all.csv', index_col=False)

In [7]:
CDC_dataset = fluProfiler_Dataset(CDC_all)
CDC_dataloader = DataLoader(CDC_dataset, batch_size=8, shuffle=False)

In [16]:
CDC_H1N1 = CDC_all.loc[CDC_all['serumType'] == 'H1N1']
CDC_H3N2 = CDC_all.loc[CDC_all['serumType'] == 'H3N2']
CDC_H1N1_dataset = fluProfiler_Dataset(CDC_H1N1)
CDC_H3N2_dataset = fluProfiler_Dataset(CDC_H3N2)
CDC_H1N1_dataloader = DataLoader(CDC_H1N1_dataset, batch_size=8, shuffle=False)
CDC_H3N2_dataloader = DataLoader(CDC_H3N2_dataset, batch_size=8, shuffle=False)

In [9]:
device = torch.device('cuda:0')

In [ ]:
import os
from tqdm import tqdm

embedding_df = CDC_all
# load embedding
sequence_names = pd.concat([embedding_df['seq_id_a'],embedding_df['seq_id_b'],
                            embedding_df['seq_id_c'],embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding", files=sequence_names)
embeddings = [emb.to(device) for emb in embeddings]
emb_dict = dict(zip(IDs, embeddings))

In [17]:
prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in CDC_H1N1_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    # matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                     matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                     matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                     labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

MAE: 0.93120
MSE: 1.41596
pearson correlation: 0.86038
spearman correlation: 0.60549
R2_score: 0.56807


(0.9311979094704648,
 1.4159646896704157,
 PearsonRResult(statistic=0.860381415636009, pvalue=0.0),
 SignificanceResult(statistic=0.6054901447566988, pvalue=0.0),
 0.5680723224865617)

In [19]:
CDC_H3N2['prediction'] = prediction_ls

/tmp/ipykernel_2955768/3578693126.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CDC_H3N2['prediction'] = prediction_ls


In [44]:
print_exams(reference_ls[2000:3000], [item + 0. for item in prediction_ls[2000:3000]])

MAE: 1.42665
MSE: 2.85267
pearson correlation: -0.21100
spearman correlation: -0.24730
R2_score: -2.23961


(1.426651471555233,
 2.8526653373803827,
 PearsonRResult(statistic=-0.21100203739144147, pvalue=1.5822279388059355e-11),
 SignificanceResult(statistic=-0.24729783184593823, pvalue=2.1217310774349147e-15),
 -2.239606765713004)

In [18]:
prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in CDC_H3N2_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    # matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                     matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                     matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                     labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

MAE: 1.38339
MSE: 3.07494
pearson correlation: 0.52536
spearman correlation: 0.52040
R2_score: -0.76623


(1.3833893413022693,
 3.0749441669639044,
 PearsonRResult(statistic=0.5253599303290183, pvalue=0.0),
 SignificanceResult(statistic=0.5204045216228483, pvalue=0.0),
 -0.7662308128735908)

In [8]:
prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    # matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                     matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                     matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                     labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

MAE:  0.5844512763892663
MSE:  0.5739428158460759
pearson correlation:  PearsonRResult(statistic=0.9219001327854721, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.9039273805361804, pvalue=0.0)
R2_score:  0.8300285395356212


In [9]:
myDF = test_data.copy()
myDF['prediction'] = prediction_ls

In [ ]:
H1N1_df = myDF[myDF['serumType'] == 'H1N1']
print_exams(H1N1_df['label'], H1N1_df['prediction'])
H3N2_df = myDF[myDF['serumType'] == 'H3N2']
print_exams(H3N2_df['label'], H3N2_df['prediction'])

MAE:  0.569174662055643
MSE:  0.5304367989270607
pearson correlation:  PearsonRResult(statistic=0.9122040877104511, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8130286921924532, pvalue=0.0)
R2_score:  0.7998475292655526
MAE:  0.6016103802867279
MSE:  0.6228099457562114
pearson correlation:  PearsonRResult(statistic=0.8973730801781679, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8962646705914706, pvalue=0.0)
R2_score:  0.7544609808665905


: 